# sweep-hparam-distribution — worked example 2: Categorical vs values — string hparams in wandb sweeps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-hparam-distribution`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When a hyperparameter is a string choice (like optimizer name or activation function), two wandb schemas work: `'values': [...]` (a simple discrete list, suitable for any type) and `'distribution': 'categorical'` with a `'values'` sub-key (explicitly marks it as categorical). Both are valid; `'values'` is more concise for small lists.

## Worked solution

**Step 1 — Use `values` for a simple list.**
For `optimizer in ['sgd', 'adam', 'adamw']`, the spec `{'values': ['sgd', 'adam', 'adamw']}` works in all three wandb search methods.

**Step 2 — Use `categorical` for explicit annotation.**
Alternatively, `{'distribution': 'categorical', 'values': ['sgd', 'adam', 'adamw']}` does the same thing but makes the distribution type explicit in the config.

**Step 3 — Never use `value` for a list.**
`'value': 'adam'` (singular) fixes the parameter to one string. Using `'value': ['sgd', 'adam']` would pass a list as the constant value, which is a schema error.

**Step 4 — Integer choices are also values.**
For `heads in [1, 2, 4, 8]`, use `{'values': [1, 2, 4, 8]}`. Integer sequences that don't form a nice uniform range belong in discrete lists, not `int_uniform`.

In [ ]:
def optimizer_spec_simple() -> dict:
    """Discrete string list — most concise form."""
    return {'values': ['sgd', 'adam', 'adamw']}

def optimizer_spec_categorical() -> dict:
    """Explicit categorical annotation — same semantics."""
    return {'distribution': 'categorical', 'values': ['sgd', 'adam', 'adamw']}

def n_heads_spec() -> dict:
    """Integer choices that are powers-of-2 — discrete list beats int_uniform."""
    return {'values': [1, 2, 4, 8]}

def fixed_activation_spec() -> dict:
    """Fixed (not swept) string — singular 'value'."""
    return {'value': 'gelu'}

# Demonstrate
print('optimizer (simple):', optimizer_spec_simple())
print('optimizer (categorical):', optimizer_spec_categorical())
print('n_heads:', n_heads_spec())
print('activation (fixed):', fixed_activation_spec())